<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 13 → Edge Analytics e Fog Computing
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

In [ ]:
def gerar_dados(n=60):
    np.random.seed(42)

    return pd.DataFrame({
        "tempo": pd.date_range("2026-10-03 08:00", periods=n, freq="1min"),
        "imagens": np.random.randint(90, 111, n),
        "suspeitas": np.random.randint(2, 9, n)
    })


df = gerar_dados()

df["sem_edge"] = df["imagens"]
df["com_edge"] = df["suspeitas"]

latencia_nuvem, latencia_edge = 420, 45
totais = [df["sem_edge"].sum(), df["com_edge"].sum()]

# 1. DADOS ENVIADOS
plt.figure(figsize=(7, 4))
plt.bar(["Sem Edge", "Com Edge"], totais)
plt.title("Dados Enviados para a Nuvem")
plt.ylabel("Quantidade de imagens")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 2. LATÊNCIA
plt.figure(figsize=(7, 4))
plt.bar(["Nuvem", "Edge"], [latencia_nuvem, latencia_edge])
plt.title("Latência para Tomada de Decisão")
plt.ylabel("Latência (ms)")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. VOLUME POR MINUTO
plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["sem_edge"], label="Sem Edge")
plt.plot(df["tempo"], df["com_edge"], label="Com Edge")
plt.title("Volume de Dados por Minuto")
plt.xlabel("Tempo")
plt.ylabel("Imagens enviadas")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. RESULTADOS
reducao = 100 * (1 - totais[1] / totais[0])

print("Imagens capturadas:", int(totais[0]))
print("Imagens enviadas com Edge:", int(totais[1]))
print(f"Redução de envio: {reducao:.1f}%")
print(f"Latência na nuvem: {latencia_nuvem} ms")
print(f"Latência no Edge: {latencia_edge} ms")

In [ ]:
categorias = [
    "Imagens capturadas", "Imagens com Edge",
    "Redução de envio (%)", "Latência nuvem (ms)",
    "Latência Edge (ms)"
]

valores = [6029, 292, 95.2, 420, 45]
cores = ["blue", "green", "orange", "red", "purple"]


def grafico():
    plt.figure(figsize=(10, 5))
    barras = plt.bar(categorias, valores, color=cores)

    plt.bar_label(barras, padding=3)

    plt.title("Resumo do Processamento de Dados na Borda (Edge)")
    plt.ylabel("Valor")
    plt.xticks(rotation=20)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

grafico()

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
df = pd.DataFrame({
    "arquitetura": ["Edge", "Fog", "Cloud"],
    "latencia_ms": [20, 80, 300],
    "dados_enviados_pct": [10, 40, 100]
})


def grafico(coluna, titulo, ylabel, cores, sufixo, limite=None):
    plt.figure(figsize=(7, 4))
    barras = plt.bar(df["arquitetura"], df[coluna], color=cores)

    plt.bar_label(
        barras,
        labels=[f"{v}{sufixo}" for v in df[coluna]],
        padding=3
    )

    plt.title(titulo)
    plt.ylabel(ylabel)
    plt.grid(axis="y", alpha=0.3)

    if limite:
        plt.ylim(limite)

    plt.tight_layout()
    plt.show()


grafico(
    "latencia_ms",
    "Latência para Tomada de Decisão",
    "Latência (ms)",
    ["green", "orange", "blue"],
    " ms"
)

grafico(
    "dados_enviados_pct",
    "Dados Enviados para Camadas Remotas",
    "Percentual dos dados (%)",
    ["purple", "red", "cyan"],
    "%",
    (0, 110)
)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
def gerar_dados():
    np.random.seed(42)

    return pd.DataFrame({
        "vibracao": np.r_[
            np.random.normal(2.0, 0.2, 150),
            np.random.normal(4.0, 0.3, 50)
        ],
        "falha_real": np.r_[np.zeros(150, int), np.ones(50, int)]
    })


df = gerar_dados()
limite = 3.0

df["predicao_edge"] = (df["vibracao"] >= limite).astype(int)
acuracia = (df["predicao_edge"] == df["falha_real"]).mean() * 100

# 1. MODELO LEVE
plt.figure(figsize=(10, 4))
plt.plot(df.index, df["vibracao"], linewidth=1.5, label="Vibração")
plt.axhline(limite, linestyle="--", linewidth=2, label="Limite = 3.0")
plt.title("Modelo Leve no Edge")
plt.xlabel("Leitura")
plt.ylabel("Vibração")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 2. COMPARAÇÃO
modelos = ["Modelo leve\n(Edge)", "Modelo pesado\n(Cloud)"]
x = np.arange(2)
largura = 0.35

plt.figure(figsize=(7, 4))
plt.bar(x - largura/2, [15, 250], largura, label="Latência (ms)")
plt.bar(x + largura/2, [20, 500], largura, label="Memória (MB)")
plt.xticks(x, modelos)
plt.title("Modelo Leve x Modelo Pesado")
plt.ylabel("Valor")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. RESULTADO
print(f"Acurácia do modelo leve: {acuracia:.1f}%")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
def grafico_barras(x, y, titulo, ylabel, cores=None, rotacao=0):
    plt.figure(figsize=(9, 4))
    plt.bar(x, y, color=cores)
    plt.title(titulo)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotacao)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


# 1. RECURSOS
df = pd.DataFrame({
    "recurso": ["CPU", "Memória", "Energia", "Rede", "Armazenamento"],
    "disponivel": [40, 30, 35, 50, 25],
    "necessario": [80, 70, 60, 65, 55]
})

x = range(len(df))
largura = 0.35

plt.figure(figsize=(9, 4))
plt.bar([i - largura/2 for i in x], df["disponivel"], largura, label="Disponível")
plt.bar([i + largura/2 for i in x], df["necessario"], largura, label="Necessário")
plt.xticks(list(x), df["recurso"])
plt.title("Limitações de Recursos no Edge")
plt.ylabel("Capacidade relativa (%)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()


# 2. DESAFIOS
desafios = pd.DataFrame({
    "desafio": ["Pouca memória", "CPU limitada", "Energia", "Rede instável", "Segurança"],
    "impacto": [85, 80, 70, 75, 90]
})

grafico_barras(
    desafios["desafio"],
    desafios["impacto"],
    "Impacto dos Principais Desafios do Edge",
    "Impacto relativo (%)",
    ["purple", "orange", "green", "blue", "red"],
    20
)